# لوحة أداء خطوط الإنتاج — نسخة Google Colab

هذي نسخة من `build_dashboard.py` مُعدّة للعمل مباشرة داخل **Google Colab**،
بدون الحاجة لأي ملفات خارجية غير `factory_production.csv`.

**الطريقة:**
1. شغّل كل خلية بالترتيب من فوق لتحت.
2. عند الوصول لخلية رفع البيانات، ارفع ملف `factory_production.csv` من جهازك
   (أو ضع رابط GitHub raw إذا كان الملف موجود في مستودعك).
3. آخر خليتين تعرضان اللوحة مباشرة داخل Colab، وتصدّرانها كملف
   `dashboard.html` تقدر تنزّله على جهازك.


---
## الخطوة 1 — تثبيت المتطلبات واستيراد المكتبات

In [1]:
!pip install jinja2 -q

import json
from pathlib import Path

import numpy as np
import pandas as pd
from jinja2 import Environment

TARGET_OEE = 75.0  # هدف الكفاءة التشغيلية المرجعي

SHIFT_ORDER = ["Morning", "Evening", "Night"]
SHIFT_LABELS_AR = {"Morning": "صباحي", "Evening": "مسائي", "Night": "ليلي"}
MONTH_NAMES_AR = {
    "01": "يناير", "02": "فبراير", "03": "مارس", "04": "أبريل",
    "05": "مايو", "06": "يونيو", "07": "يوليو", "08": "أغسطس",
    "09": "سبتمبر", "10": "أكتوبر", "11": "نوفمبر", "12": "ديسمبر",
}


---
## الخطوة 2 — ارفع بيانات `factory_production.csv`

شغّل الخلية التالية. إذا كنت داخل Colab بتظهر لك زر رفع ملف — اختر
`factory_production.csv` من جهازك. إذا كنت تشغّل الدفتر خارج Colab (محليًا)
وعندك الملف بنفس مجلد الدفتر، بيتم تخطي الرفع تلقائيًا.

In [2]:
DATA_PATH = Path("factory_production.csv")

if not DATA_PATH.exists():
    try:
        from google.colab import files
        print("ارفع ملف factory_production.csv:")
        uploaded = files.upload()
        # يأخذ أول ملف مرفوع ويحفظه بالاسم المتوقع
        first_name = next(iter(uploaded))
        Path(first_name).rename(DATA_PATH) if first_name != DATA_PATH.name else None
    except ImportError:
        raise FileNotFoundError(
            "ضع ملف factory_production.csv بجانب هذا الدفتر ثم أعد التشغيل."
        )

print("تم العثور على الملف:", DATA_PATH.resolve())


تم العثور على الملف: /content/factory_production.csv


---
## الخطوة 3 — حمّل البيانات واحسب كل الأرقام (قبل أي رسم)

In [3]:
df = pd.read_csv(DATA_PATH, parse_dates=["date"])
df["shift"] = pd.Categorical(df["shift"], categories=SHIFT_ORDER, ordered=True)
df.head()


,date,line_id,shift,product,planned_units,units_produced,units_defective,defect_rate_pct,oee_pct,planned_downtime_min,unplanned_downtime_min,machine_temp_c,energy_kwh,operator_count,scrap_cost_sar
0,2025-01-01,Line-A,Morning,Product-102,1146,932,16,1.76,82.75,39,4.6,67.5,658.0,3,650.15
1,2025-01-01,Line-A,Evening,Product-101,1278,1018,16,1.59,80.89,31,19.8,63.7,619.5,5,716.78
2,2025-01-01,Line-A,Night,Product-205,1307,939,27,2.88,71.40,24,24.0,69.6,543.5,6,1214.71
3,2025-01-01,Line-B,Morning,Product-102,1350,986,26,2.64,74.17,22,13.8,72.6,603.3,3,730.62
4,2025-01-01,Line-B,Evening,Product-205,1120,853,21,2.57,74.54,26,10.1,69.6,644.3,4,894.83


In [4]:
def build_ledger(df: pd.DataFrame) -> list[dict]:
    total_units = int(df["units_produced"].sum())
    total_defective = int(df["units_defective"].sum())
    national_oee = round(np.average(df["oee_pct"], weights=df["units_produced"]), 1)
    avg_defect_rate = round(df["defect_rate_pct"].mean(), 2)
    total_scrap_cost = int(round(df["scrap_cost_sar"].sum(), 0))

    return [
        {"value": f"{len(df):,}", "label": "سجل تشغيل يومي", "cls": ""},
        {"value": f"{national_oee}%", "label": "الكفاءة التشغيلية الوطنية (OEE)",
         "cls": "good" if national_oee >= TARGET_OEE else "bad"},
        {"value": f"{avg_defect_rate}%", "label": "متوسط معدل العيوب", "cls": "bad"},
        {"value": f"{total_units:,}", "label": "وحدة منتجة", "cls": ""},
        {"value": f"{total_defective:,}", "label": "وحدة معيبة", "cls": "bad"},
        {"value": f"{total_scrap_cost:,}", "label": "تكلفة الهدر (ريال)", "cls": "bad"},
    ]


def build_trend(df: pd.DataFrame) -> dict:
    d = df.copy()
    d["month"] = d["date"].dt.strftime("%Y-%m")
    pivot = (
        d.groupby(["month", "shift"], as_index=False, observed=True)["oee_pct"]
        .mean()
        .pivot(index="month", columns="shift", values="oee_pct")
        .sort_index()
    )
    months_ar = [f"{MONTH_NAMES_AR[m.split('-')[1]]}" for m in pivot.index]
    return {
        "months": months_ar,
        "morning": [round(v, 1) for v in pivot["Morning"]],
        "evening": [round(v, 1) for v in pivot["Evening"]],
        "night": [round(v, 1) for v in pivot["Night"]],
    }


def build_lines(df: pd.DataFrame) -> dict:
    by_line = (
        df.groupby("line_id", as_index=False)["oee_pct"].mean().sort_values("oee_pct")
    )
    colors = ["#B23A2E" if v < TARGET_OEE else "#1F6F64" for v in by_line["oee_pct"]]
    return {
        "labels": by_line["line_id"].tolist(),
        "values": [round(v, 1) for v in by_line["oee_pct"]],
        "colors": colors,
    }


def build_products(df: pd.DataFrame) -> dict:
    by_product = (
        df.groupby("product", as_index=False)["defect_rate_pct"]
        .mean()
        .sort_values("defect_rate_pct", ascending=False)
    )
    return {
        "labels": by_product["product"].tolist(),
        "values": [round(v, 2) for v in by_product["defect_rate_pct"]],
    }


def build_downtime(df: pd.DataFrame) -> dict:
    by_shift = (
        df.groupby("shift", as_index=False, observed=True)["unplanned_downtime_min"]
        .mean()
        .sort_values("unplanned_downtime_min")
    )
    return {
        "labels": [SHIFT_LABELS_AR[s] for s in by_shift["shift"]],
        "values": [round(v, 1) for v in by_shift["unplanned_downtime_min"]],
    }


def build_scrap_combos(df: pd.DataFrame, top_n: int = 15) -> dict:
    combo = (
        df.groupby(["line_id", "shift"], as_index=False, observed=True)["scrap_cost_sar"]
        .sum()
        .sort_values("scrap_cost_sar", ascending=False)
        .head(top_n)
    )
    labels = [
        f"{row.line_id} · {SHIFT_LABELS_AR[row.shift]}" for row in combo.itertuples()
    ]
    return {
        "labels": labels,
        "values": [round(v, 0) for v in combo["scrap_cost_sar"]],
    }


def build_worst_records(df: pd.DataFrame, top_n: int = 12) -> list[dict]:
    worst = df.sort_values("defect_rate_pct", ascending=False).head(top_n)
    return [
        {
            "date": row.date.strftime("%Y-%m-%d"),
            "line_id": row.line_id,
            "shift": SHIFT_LABELS_AR[row.shift],
            "product": row.product,
            "defect_rate_pct": round(row.defect_rate_pct, 2),
        }
        for row in worst.itertuples()
    ]


In [5]:
ledger = build_ledger(df)
trend = build_trend(df)
lines = build_lines(df)
products = build_products(df)
downtime = build_downtime(df)
scrap = build_scrap_combos(df)
worst_records = build_worst_records(df)

print("الأرقام الأساسية:")
for item in ledger:
    print(f"  - {item['label']}: {item['value']}")


الأرقام الأساسية:
  - سجل تشغيل يومي: 2,715
  - الكفاءة التشغيلية الوطنية (OEE): 74.9%
  - متوسط معدل العيوب: 2.54%
  - وحدة منتجة: 2,219,170
  - وحدة معيبة: 54,489
  - تكلفة الهدر (ريال): 2,258,728


لاحظ إننا حسبنا وطبعنا كل الأرقام **قبل** بناء أي رسم — نفس مبدأ الدفتر
الأصلي. لو رقم غريب طلع هنا (OEE سالب، تكلفة هدر صفرية...) تكتشفه الحين.

---
## الخطوة 4 — القالب (HTML/CSS/Chart.js)

نفس تصميم اللوحة السابقة (سجل علوي، أقسام بخط فاصل، Chart.js للرسوم)،
مُضمَّن هنا كنص بايثون عشان الدفتر يكون ملفًا واحدًا مستقلًا لا يحتاج قوالب خارجية.

In [6]:
TEMPLATE_HTML = """<!DOCTYPE html>
<html lang="ar" dir="rtl">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>لوحة أداء خطوط الإنتاج</title>
<link rel="preconnect" href="https://fonts.googleapis.com">
<link href="https://fonts.googleapis.com/css2?family=Fraunces:opsz,wght@9..144,400;9..144,500;9..144,600;9..144,700&family=IBM+Plex+Sans:wght@400;500;600&family=IBM+Plex+Sans+Arabic:wght@400;500;600;700&display=swap" rel="stylesheet">
<script src="https://cdnjs.cloudflare.com/ajax/libs/Chart.js/4.4.4/chart.umd.min.js"></script>
<style>
  :root{
    --paper:#F1F2ED;
    --paper-raised:#FBFBF8;
    --ink:#15201C;
    --ink-soft:#4B564F;
    --line:#D7DAD0;
    --bad:#B23A2E;
    --bad-soft:#E7C3BC;
    --good:#1F6F64;
    --good-soft:#C4DDD7;
    --warn:#8E6A2E;
    --warn-soft:#E4D3B3;
    --risk1:#5C7A63;
    --risk2:#C98A3B;
    --risk3:#B23A2E;
  }
  *{box-sizing:border-box;}
  html,body{margin:0;padding:0;}
  body{
    background:var(--paper);
    color:var(--ink);
    font-family:'IBM Plex Sans Arabic','IBM Plex Sans',sans-serif;
    line-height:1.6;
    -webkit-font-smoothing:antialiased;
  }
  .wrap{max-width:1120px;margin:0 auto;padding:0 28px 100px;}
  .en{font-family:'IBM Plex Sans',sans-serif;direction:ltr;unicode-bidi:isolate;}

  header.masthead{border-bottom:2px solid var(--ink);padding:44px 0 22px;margin-bottom:8px;}
  .masthead-top{display:flex;justify-content:space-between;align-items:baseline;flex-wrap:wrap;gap:10px;margin-bottom:18px;}
  .ref-tag{font-family:'IBM Plex Sans',sans-serif;font-size:12.5px;color:var(--ink-soft);direction:ltr;letter-spacing:0.02em;}
  h1{font-family:'Fraunces',serif;font-weight:600;font-size:clamp(30px,4.4vw,46px);line-height:1.15;margin:0 0 14px;max-width:18ch;}
  .dek{font-size:16.5px;color:var(--ink-soft);max-width:64ch;margin:0;}
  .dek .en{font-weight:500;color:var(--ink);}

  .ledger{display:flex;flex-wrap:wrap;border-top:1px solid var(--line);border-bottom:1px solid var(--line);margin:30px 0 46px;padding:22px 0;}
  .ledger-item{flex:1 1 150px;padding:0 22px;border-inline-start:1px solid var(--line);}
  .ledger-item:first-child{border-inline-start:none;padding-inline-start:0;}
  .ledger-num{font-family:'Fraunces',serif;font-weight:600;font-size:28px;direction:ltr;display:block;}
  .ledger-num.bad{color:var(--bad);}
  .ledger-num.good{color:var(--good);}
  .ledger-label{font-size:13px;color:var(--ink-soft);margin-top:3px;display:block;}

  section{margin:56px 0;}
  .section-head{display:flex;justify-content:space-between;align-items:flex-end;gap:16px;margin-bottom:6px;border-bottom:1px solid var(--ink);padding-bottom:10px;}
  h2{font-family:'Fraunces',serif;font-weight:600;font-size:23px;margin:0;}
  .section-note{font-size:13px;color:var(--ink-soft);max-width:44ch;text-align:start;}
  p.lede{color:var(--ink-soft);font-size:15px;max-width:70ch;margin:14px 0 22px;}

  .panel{background:var(--paper-raised);border:1px solid var(--line);border-radius:3px;padding:22px 22px 14px;}
  .grid-2{display:grid;grid-template-columns:1fr 1fr;gap:22px;}
  @media (max-width:760px){.grid-2{grid-template-columns:1fr;}}
  canvas{max-width:100%;}

  table{width:100%;border-collapse:collapse;font-size:14px;}
  th{text-align:start;font-weight:500;color:var(--ink-soft);font-size:12.5px;padding:8px 10px;border-bottom:1px solid var(--ink);}
  td{padding:9px 10px;border-bottom:1px solid var(--line);}
  tr:last-child td{border-bottom:none;}
  td.num{direction:ltr;text-align:end;font-family:'IBM Plex Sans',sans-serif;color:var(--bad);font-weight:500;}
  td.tag{font-family:'IBM Plex Sans',sans-serif;direction:ltr;text-align:start;unicode-bidi:isolate;}

  footer{margin-top:70px;padding-top:22px;border-top:1px solid var(--line);font-size:12.5px;color:var(--ink-soft);}
</style>
</head>
<body>
<div class="wrap">

  <header class="masthead">
    <div class="masthead-top">
      <span class="ref-tag">FACTORY PERFORMANCE REGISTER · {{ date_range_en }}</span>
      <span class="ref-tag">مصدر البيانات: factory_production.csv</span>
    </div>
    <h1>لوحة أداء خطوط الإنتاج</h1>
    <p class="dek">قراءة في <span class="en">{{ total_records_fmt }}</span> سجل تشغيل يوميّ عبر <span class="en">{{ n_lines }}</span> خطوط إنتاج و<span class="en">{{ n_products }}</span> منتجات، على مدى {{ n_months }} أشهر من <span class="en">{{ date_min_ar }}</span> إلى <span class="en">{{ date_max_ar }}</span>.</p>
  </header>

  <div class="ledger">
    {% for item in ledger %}
    <div class="ledger-item">
      <span class="ledger-num {{ item.cls }} en">{{ item.value }}</span>
      <span class="ledger-label">{{ item.label }}</span>
    </div>
    {% endfor %}
  </div>

  <section id="sec-trend">
    <div class="section-head">
      <h2>اتجاه الكفاءة التشغيلية (OEE) حسب الوردية</h2>
      <span class="section-note">متوسط شهري لكل وردية — الوردية الليلية هي الأضعف باستمرار</span>
    </div>
    <p class="lede">الكفاءة التشغيلية (OEE) لكل الورديات في تراجع تدريجي منذ يناير، وتحديدًا في وردية الليل التي تبقى أدنى من الهدف ({{ target_oee }}%) طوال الفترة.</p>
    <div class="panel"><canvas id="chartTrend" height="90"></canvas></div>
  </section>

  <section id="sec-lines-products">
    <div class="grid-2">
      <div>
        <div class="section-head"><h2>خطوط الإنتاج حسب الكفاءة</h2></div>
        <p class="lede">متوسط OEE لكل خط إنتاج على كامل الفترة. الخطوط باللون الأحمر تحت الهدف المحدد ({{ target_oee }}%).</p>
        <div class="panel"><canvas id="chartLines" height="230"></canvas></div>
      </div>
      <div>
        <div class="section-head"><h2>معدل العيوب حسب المنتج</h2></div>
        <p class="lede">متوسط نسبة الوحدات المعيبة لكل منتج. الفروقات بين المنتجات محدودة نسبيًا مقارنة بالفروقات بين الخطوط.</p>
        <div class="panel"><canvas id="chartProducts" height="230"></canvas></div>
      </div>
    </div>
  </section>

  <section id="sec-downtime">
    <div class="section-head">
      <h2>متوسط التوقف غير المخطط حسب الوردية</h2>
      <span class="section-note">بالدقيقة لكل سجل تشغيل</span>
    </div>
    <p class="lede">وردية الليل تسجل أعلى متوسط توقف غير مخطط، وهو ما يفسّر جزئيًا انخفاض كفاءتها التشغيلية مقارنة بالورديتين الأخريين.</p>
    <div class="panel"><canvas id="chartDowntime" height="80"></canvas></div>
  </section>

  <section id="sec-scrap">
    <div class="section-head">
      <h2>أعلى تركيبات (خط × وردية) من حيث تكلفة الهدر</h2>
      <span class="section-note">إجمالي تكلفة الهدر بالريال السعودي — أعلى {{ top_combos|length }} تركيبة</span>
    </div>
    <p class="lede">ترتيب كل مزيج من خط الإنتاج والوردية حسب إجمالي تكلفة الهدر المتراكمة على مدى الفترة، لتحديد أولويات التدخل التشغيلي.</p>
    <div class="panel"><canvas id="chartScrap" height="290"></canvas></div>
  </section>

  <section id="sec-worst">
    <div class="section-head">
      <h2>أعلى السجلات من حيث معدل العيوب</h2>
      <span class="section-note">أسوأ {{ worst_records|length }} سجل تشغيل يومي</span>
    </div>
    <div class="panel">
      <table>
        <thead><tr><th>التاريخ</th><th>الخط</th><th>الوردية</th><th>المنتج</th><th style="text-align:end">معدل العيوب</th></tr></thead>
        <tbody>
          {% for r in worst_records %}
          <tr>
            <td class="tag">{{ r.date }}</td>
            <td class="tag">{{ r.line_id }}</td>
            <td class="tag">{{ r.shift }}</td>
            <td class="tag">{{ r.product }}</td>
            <td class="num">{{ r.defect_rate_pct }}%</td>
          </tr>
          {% endfor %}
        </tbody>
      </table>
    </div>
  </section>

  <footer>
    مصدر البيانات: <span class="en">factory_production.csv</span> — {{ total_records_fmt }} سجل تشغيل يومي لكل تركيبة (خط × وردية × منتج) بين <span class="en">{{ date_min_ar }}</span> و<span class="en">{{ date_max_ar }}</span>. هذه اللوحة مولّدة تلقائيًا عبر <span class="en">build_dashboard.py</span>.
  </footer>

</div>

<script>
const DATA = {{ chart_data_json | safe }};

Chart.defaults.font.family = "'IBM Plex Sans', sans-serif";
Chart.defaults.color = '#4B564F';
Chart.defaults.borderColor = '#D7DAD0';
const gridColor = '#E4E6DF';

// Trend: OEE by month per shift
new Chart(document.getElementById('chartTrend'), {
  type:'line',
  data:{
    labels: DATA.trend.months,
    datasets:[
      {label:'صباحي', data: DATA.trend.morning, borderColor:'#1F6F64', backgroundColor:'#1F6F64', tension:0.3, pointRadius:3},
      {label:'مسائي', data: DATA.trend.evening, borderColor:'#8E6A2E', backgroundColor:'#8E6A2E', tension:0.3, pointRadius:3},
      {label:'ليلي', data: DATA.trend.night, borderColor:'#B23A2E', backgroundColor:'#B23A2E', tension:0.3, pointRadius:3},
    ]
  },
  options:{
    responsive:true,
    plugins:{ legend:{ position:'top', align:'end', labels:{boxWidth:10, boxHeight:10, usePointStyle:true, pointStyle:'circle'} } },
    scales:{ x:{ grid:{display:false} }, y:{ grid:{color:gridColor}, title:{display:true, text:'OEE %'} } }
  }
});

// Lines by OEE
new Chart(document.getElementById('chartLines'), {
  type:'bar',
  data:{
    labels: DATA.lines.labels,
    datasets:[{ data: DATA.lines.values, backgroundColor: DATA.lines.colors, borderRadius:2, maxBarThickness:26 }]
  },
  options:{
    indexAxis:'y',
    plugins:{ legend:{display:false} },
    scales:{ x:{ grid:{color:gridColor}, title:{display:true, text:'OEE %'} }, y:{ grid:{display:false} } }
  }
});

// Products by defect rate
new Chart(document.getElementById('chartProducts'), {
  type:'bar',
  data:{
    labels: DATA.products.labels,
    datasets:[{ data: DATA.products.values, backgroundColor:'#B23A2E', borderRadius:2, maxBarThickness:26 }]
  },
  options:{
    indexAxis:'y',
    plugins:{ legend:{display:false} },
    scales:{ x:{ grid:{color:gridColor}, title:{display:true, text:'معدل العيوب %'} }, y:{ grid:{display:false} } }
  }
});

// Downtime by shift
new Chart(document.getElementById('chartDowntime'), {
  type:'bar',
  data:{
    labels: DATA.downtime.labels,
    datasets:[{ data: DATA.downtime.values, backgroundColor:['#5C7A63','#C98A3B','#B23A2E'], borderRadius:2, maxBarThickness:46 }]
  },
  options:{
    indexAxis:'y',
    plugins:{ legend:{display:false} },
    scales:{ x:{ grid:{color:gridColor}, title:{display:true, text:'دقيقة'} }, y:{ grid:{display:false} } }
  }
});

// Top line x shift combos by scrap cost
new Chart(document.getElementById('chartScrap'), {
  type:'bar',
  data:{
    labels: DATA.scrap.labels,
    datasets:[{ data: DATA.scrap.values, backgroundColor:'#1F6F64', borderRadius:2, maxBarThickness:18 }]
  },
  options:{
    indexAxis:'y',
    plugins:{ legend:{display:false} },
    scales:{ x:{ grid:{color:gridColor}, title:{display:true, text:'تكلفة الهدر (ريال سعودي)'} }, y:{ grid:{display:false} } }
  }
});
</script>
</body>
</html>
"""


---
## الخطوة 5 — اربط الأرقام بالقالب وابنِ ملف HTML

In [7]:
chart_data = {
    "trend": trend,
    "lines": lines,
    "products": products,
    "downtime": downtime,
    "scrap": scrap,
}

context = {
    "date_range_en": f"{df['date'].min():%b %Y} – {df['date'].max():%b %Y}",
    "date_min_ar": df["date"].min().strftime("%Y-%m-%d"),
    "date_max_ar": df["date"].max().strftime("%Y-%m-%d"),
    "n_months": df["date"].dt.to_period("M").nunique(),
    "n_lines": df["line_id"].nunique(),
    "n_products": df["product"].nunique(),
    "total_records_fmt": f"{len(df):,}",
    "target_oee": int(TARGET_OEE),
    "ledger": ledger,
    "top_combos": scrap["labels"],
    "worst_records": worst_records,
    "chart_data_json": json.dumps(chart_data, ensure_ascii=False),
}

env = Environment()
template = env.from_string(TEMPLATE_HTML)
html = template.render(**context)

print(f"تم بناء اللوحة — الحجم: {len(html):,} حرف")


تم بناء اللوحة — الحجم: 14,953 حرف


---
## الخطوة 6 — اعرض اللوحة مباشرة داخل Colab

In [8]:
from IPython.display import HTML, display

display(HTML(html))


التاريخ,الخط,الوردية,المنتج,معدل العيوب
2025-06-17,Line-C,صباحي,Product-205,4.72%
2025-06-29,Line-C,ليلي,Product-101,4.71%
2025-05-25,Line-C,ليلي,Product-101,4.62%
2025-06-06,Line-C,ليلي,Product-101,4.59%
2025-04-10,Line-E,ليلي,Product-101,4.53%
2025-05-31,Line-C,ليلي,Product-101,4.46%
2025-05-15,Line-B,مسائي,Product-102,4.43%
2025-03-24,Line-E,ليلي,Product-101,4.42%
2025-05-08,Line-E,مسائي,Product-102,4.42%
2025-06-04,Line-C,مسائي,Product-101,4.39%


---
## الخطوة 7 — احفظ اللوحة ونزّلها كملف مستقل

الملف الناتج `dashboard.html` يفتح في أي متصفح بدون Python أو Colab.

In [9]:
output_path = Path("dashboard.html")
output_path.write_text(html, encoding="utf-8")
print(f"تم الحفظ: {output_path.resolve()}")

try:
    from google.colab import files
    files.download(str(output_path))
except ImportError:
    print("نزّل الملف يدويًا من مجلد الدفتر إذا كنت تعمل خارج Colab.")


تم الحفظ: /content/dashboard.html


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>